# broadcast-source-fanout — worked example 3: Backprop through a codebook fan-out counts each row's usage

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcast-source-fanout`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

When you fan a `(K, D)` source table out by indexing with code ids and then sum the result, autograd accumulates gradient back into each source row once per time that row was selected. So a `sum()`-loss gradient on the codebook equals, per row, the number of times that code id appeared — and rows never selected get a structural zero. This mirrors how a broadcast's reduce-counterpart accumulates contributions from every destination back to the source.

## Worked solution

**Goal.** Build a leaf codebook, fan it out by `codes`, take `sum()` as the loss, backprop, and show the codebook gradient row-norms count usage.

1. **Make the codebook a leaf requiring grad.** `codebook = t.arange(...).reshape(K, D).requires_grad_(True)` gives unique, inspectable rows and a tracked leaf so `.grad` will populate.
2. **Fan out by indexing.** `fanned = codebook[codes]` shape `(B, L, D)`. Each output element is a copy of a source row, and autograd records which source row each came from.
3. **Scalar loss.** `loss = fanned.sum()` differentiates to 1 for every element of `fanned`. Because each `fanned` element copies one codebook element, the gradient routed back to codebook element `(k, d)` is the *count* of fan-outs that read it.
4. **Backward and inspect.** After `loss.backward()`, every row of `codebook.grad` is a constant vector whose value equals how many times that code id appeared in `codes`. Absent ids give an exactly-zero row (autograd never touched them).
5. **Why this matches a broadcast.** Broadcasting fans one source to many destinations; the adjoint (reduce/gather) sums the many destinations' contributions back into the one source. The per-row count is precisely that summed adjoint.

In [ ]:
import torch as t
from torch import Tensor

def codebook_usage_grad(K: int, D: int, codes: Tensor):
    codebook = t.arange(K * D, dtype=t.float32).reshape(K, D).requires_grad_(True)
    fanned = codebook[codes]          # (B, L, D) fan-out
    loss = fanned.sum()
    loss.backward()
    return codebook.grad.clone()

t.manual_seed(0)
K, D = 4, 3
codes = t.tensor([[0, 0, 2], [2, 2, 0]])   # id 0 x3, id 2 x3, ids 1&3 absent
grad = codebook_usage_grad(K, D, codes)
print("grad row counts:", grad[:, 0].tolist())
print("absent rows zero:", bool((grad[1] == 0).all() and (grad[3] == 0).all()))